# Day 1 — Extra exercises: measurements, summaries and probability

**For two kinds of reader.** If you **missed Day 1** (after informing the lecturer in advance), this notebook is part of your make-up work. If you attended and want **more practice**, work through any part at your own pace; nothing here is required for you.

**What a make-up submission for Day 1 contains** (see `../day-1-makeup.md`)

- **A1–A3 and B1–B3** of this notebook;
- your own solutions to the core in-class exercises in `../day-1-exercises-morning.ipynb` and `../day-1-exercises-afternoon.ipynb`.

The reflection prompts below are optional. **C-problems are optional challenges** for everyone. The make-up submission is graded on the 0–4 in-class scale and counts as your in-class score for the missed day.

Topics: Part A — data structure, summaries and plots of a repeated-measurement dataset. Part B — Bayes' rule with discrete hypotheses, densities versus probabilities, expectation and a first prediction. Work individually after looking through the Day 1 slides (`../day-1-slides-morning.pdf` and `../day-1-slides-afternoon.pdf`).

**How to work.** Open in JupyterLab, select **R (bayes)**, run cells in order from a fresh kernel. Hints are hidden under **Hint**; try first. Write explanations in your own words in the *Your answer* cells — numbers alone are not enough. Worked solutions are released only after the make-up deadline. All data are **synthetic teaching data**; never report them as real measurements.

**Submitting (make-up).** Submit this notebook with outputs, any `.stan` files you edited, and your written answers, together with the in-class exercises, through the announced route by the announced deadline (normally within one week of the missed day). Discussing ideas is fine; the code and explanations must be your own. You may use AI assistants without declaring it, but you should be able to explain every line.

## Setup

R with the tidyverse: dplyr for summaries and ggplot2 for figures, as in the class notebooks. No Stan today. Run this cell first.

In [ ]:
library(tidyverse)
set.seed(2026)
options(width=100, repr.plot.width=8, repr.plot.height=5)
DATA_DIR <- "data"
q90 <- function(x) quantile(x, c(.05, .5, .95))

## Part A — A repeated-measurement dataset

A laboratory checks its analytical balance every day by weighing a certified **50.000 g reference mass**. Five operators (O1–O5) each weighed the mass **eight times**, four times in the morning session and four in the afternoon. The file `data/weighing-synthetic.csv` has columns:

- `operator` — anonymous operator ID;
- `replicate` — weighing number 1–8 for that operator;
- `session` — morning (replicates 1–4) or afternoon (5–8);
- `mass_g` — the displayed mass in grams, or `NA` if no reading was obtained.

The scientific question: *is the balance-plus-operator measurement process accurate (centred on 50.000 g) and how variable is it?*

### A1. What is in the data? (25 min)

Read the file and report: (a) the number of rows, operators and missing readings; (b) the **observational unit** (what one row is); (c) why the 40 rows are *not* 40 independent pieces of information about the balance; (d) a derived variable that makes the numbers easier to read. Create `dev_mg`, the deviation from 50 g in **milligrams**.

In [ ]:
w <- read.csv(file.path(DATA_DIR, "weighing-synthetic.csv"))
str(w)
# (a) rows, operators, missing readings
# (d) deviation from the reference in milligrams:
# w <- w |> mutate(dev_mg = ...)

<details><summary><b>Hint</b> (click to open)</summary>

`nrow(w)` counts rows, `n_distinct(w$operator)` counts distinct operators and `sum(is.na(w$mass_g))` counts missing values. `mutate()` adds a new column. 1 g = 1000 mg.

</details>

**Your answer:** *(write here)*

### A2. Summaries and an unusual value (30 min)

(a) Compute the pooled mean, median and SD of `dev_mg` (ignoring the missing value explicitly). (b) Compute a per-operator table with n, mean, median and SD. (c) One reading looks very different from the others. Find it, and suggest a plausible explanation by looking at the digits. (d) Recompute the pooled mean and SD **without** that reading as a *sensitivity analysis*, keeping the original data unchanged. Which summaries change a lot, and which hardly change?

In [ ]:
# (a) pooled summaries; na.rm = TRUE ignores the missing reading
# w |> summarise(n = sum(!is.na(dev_mg)), mean = mean(dev_mg, na.rm = TRUE), median = ..., sd = ...)
# (b) per-operator table: the same summarise() after group_by(operator)
# (c) find the unusual reading: arrange(desc(abs(dev_mg)))
# (d) sensitivity: summaries with the reading excluded, keeping w unchanged
# ws <- w |> filter(!(operator == "O5" & replicate == 3))

<details><summary><b>Hint</b> (click to open)</summary>

`w |> arrange(desc(abs(dev_mg))) |> head(3)` shows the three rows furthest from 0. Compare the recorded value 50.120 g with the other O5 readings — which digit might have been mistyped?

</details>

**Your answer:** *(write here)*

### A3. Two useful plots (20 min)

Make (a) a pooled histogram of `dev_mg` and (b) a plot showing each operator's readings separately (for example jittered points with `geom_jitter()`, or boxplots with `geom_boxplot()`) with a reference line at 0 mg (`geom_vline()` or `geom_hline()`). Use the sensitivity version without the unusual reading for (b) if the scale becomes unreadable, and say so in the caption. What does the per-operator plot show that the pooled histogram hides? Is there evidence of a morning/afternoon difference?

In [ ]:
# (a) pooled histogram with labelled axes and units
ggplot(w, aes(x = dev_mg)) +
  geom_histogram(bins = 20) +
  labs(title = "...", x = "...", y = "...")

# (b) per-operator plot, for example:
# ggplot(ws, aes(x = dev_mg, y = operator, shape = session)) + geom_vline(xintercept = 0, linetype = 2) + geom_jitter(height = 0.15, width = 0) + labs(...)

<details><summary><b>Hint</b> (click to open)</summary>

`shape = session` inside `aes()` gives morning and afternoon readings different symbols; `scale_shape_manual(values = c(morning = 1, afternoon = 19))` makes them open and filled. ggplot2 warns that it removed the row with the missing reading: mention it in your caption.

</details>

**Your answer:** *(write here)*

### Reflection A (optional)

In 80–120 words: distinguish **variability between weighings by the same operator**, **differences between operators** and **uncertainty about the balance's true offset**. Which of these would more data from the same five operators reduce?

**Your answer:** *(write here)*

## Part B — Probability and updating

### B1. Which machine? Bayes' rule with two hypotheses (30 min)

A workshop receives parts from two machines. Machine A makes **70 %** of parts and **2 %** of its parts are defective; machine B makes **30 %** and **10 %** of its parts are defective. A box contains parts that all come from **one** machine, but the label is lost.

(a) You test one part from the box and it is defective. What is the probability the box came from machine B? Set it up as *prior × likelihood, then normalise*, as in the marble example. (b) A second part from the same box is also defective. Update again, using (a) as the new prior. Check that you get the same answer by updating once with both observations. (c) What is the probability that a **third** part from this box is defective? (d) Check (a) by simulation: simulate 100,000 boxes and one tested part each.

In [ ]:
prior <- c(A = 0.7, B = 0.3)
p_def <- c(A = 0.02, B = 0.10)
# (a) post1 <- prior * likelihood / sum(prior * likelihood)
# (b) update again; compare with a single update using p_def^2
# (c) predictive probability: sum over machines of P(machine | data) * P(defective | machine)
# (d) simulation check

<details><summary><b>Hint</b> (click to open)</summary>

The likelihood of "defective" is 0.02 under A and 0.10 under B. After two defectives the likelihood is 0.02² and 0.10² (independent parts given the machine). In (d), keep only the simulated boxes whose tested part was defective.

</details>

**Your answer:** *(write here)*

### B2. Densities are not probabilities (25 min)

The time *T* (minutes) between two requests arriving at a shared lab instrument is modelled as **exponential with mean 6 minutes** (rate 1/6 per minute).

(a) Evaluate the density at *t* = 0 and *t* = 3 and give its units. (b) Compute P(*T* < 3), P(2 < *T* < 4) and P(*T* = 3). (c) Find the median waiting time and compare it with the mean. (d) A different quantity *U* is uniform on [0, 0.25] hours. What is its density, and why is a density larger than 1 not a contradiction?

In [ ]:
rate <- 1/6
# (a) dexp(c(0, 3), rate)
# (b) pexp() differences
# (c) qexp(0.5, rate)
# (d) dunif(0.1, 0, 0.25); punif(0.25, 0, 0.25)

<details><summary><b>Hint</b> (click to open)</summary>

For a continuous variable, probabilities are **areas** under the density: P(a < T < b) = F(b) − F(a), where F is `pexp`. The probability of any single exact value is 0.

</details>

**Your answer:** *(write here)*

### B3. Expectation, variance and a first prediction (25 min)

(a) Let *X* be the number of defective parts in a box of 5 from machine B (each defective with probability 0.10, independently). Tabulate the PMF, then compute E[*X*] and Var(*X*) **from the table** (as sums), and compare with *np* and *np*(1−*p*). Check both by simulation. (b) Return to the weighing data. Give an informal central prediction and a plausible range for (i) **the next weighing by O4** and (ii) **the first weighing by a new operator** who has not been observed. Explain why (ii) should be wider. This is informal reasoning; we will build proper Bayesian predictive distributions for such questions on Days 2 and 4.

In [ ]:
x <- 0:5
px <- dbinom(x, 5, 0.10)
# (a) E = sum(x * px); Var = sum((x - E)^2 * px); simulation with rbinom()
# (b) use your per-operator table from A2

<details><summary><b>Hint</b> (click to open)</summary>

For (b)(ii), think about two sources of variation: where a new operator's own average might lie (look at how the five operator means differ) *and* the spread of single weighings around that average.

</details>

**Your answer:** *(write here)*

### Reflection B (optional)

In 80–120 words: explain the difference between P(B | defective) and P(defective | B) using the numbers in B1, and say in one sentence why a predictive probability (B1c) lies between the machine-specific defect rates.

**Your answer:** *(write here)*

## C. Optional challenges

**C1.** In B1, how many consecutive defective parts would be needed before P(B | data) exceeds 0.99? Find the answer with a short loop and explain why the answer depends strongly on the prior 0.3.

**C2.** For the weighing data, compute each operator's mean and a rough standard error SD/√n. Is O4's mean distinguishable from 0 mg by this rough measure? Why does this approach ignore something important about a *new* operator?

**C3.** Show by simulation that the **median** of 9 exponential waiting times (mean 6) is a more stable summary than the **mean** when one reading is occasionally recorded in seconds instead of minutes (multiply a randomly chosen value by 60 in 10 % of simulated datasets).

In [ ]:
# Optional challenge work

**Your answer:** *(write here)*

## Make-up checklist

- [ ] A1–A3 and B1–B3 · [ ] in-class exercises (`../day-1-exercises-morning.ipynb` and `../day-1-exercises-afternoon.ipynb`)
- [ ] Notebook runs from a fresh kernel; plots have labelled axes with units
- [ ] Every answer cell contains your own explanation, not only numbers
- [ ] I can explain every line of my code